In [ ]:
##Train Script

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tqdm import tqdm

# =====================================================
# 1️⃣ CONFIGURATION
# =====================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {DEVICE}")

OUTPUT_DIR = "windows_new"
MODEL_PATH = "model.pt"
BATCH_SIZE = 64
EPOCHS = 500
GRAD_CLIP = 0.5
LEARNING_RATE = 5e-5  
DROPOUT = 0.1

# =====================================================
# 2️⃣ LOAD PREPROCESSED TRAIN DATA
# =====================================================
train = pickle.load(open(os.path.join(OUTPUT_DIR, "train_windows.pkl"), "rb"))
companies = pickle.load(open(os.path.join(OUTPUT_DIR, "train_company_list.pkl"), "rb"))
print("✅ Loaded 'train_windows.pkl' and 'train_company_list.pkl'.")

# =====================================================
# 3️⃣ ENCODE COMPANY LABELS
# =====================================================
enc = LabelEncoder().fit(companies)
train_encoded = []
y_scalers = {}

for X, next_s, y, company in train:
    company_idx = enc.transform([company])[0]

    # Scale target y using MinMaxScaler per company
    if company not in y_scalers:
        y_scaler = MinMaxScaler()
        # fit scaler on all y-values of this company
        company_y = [yi for _, _, yi, c in train if c == company]
        y_scaler.fit(np.array(company_y).reshape(-1,1))
        y_scalers[company] = y_scaler
    else:
        y_scaler = y_scalers[company]

    y_scaled = y_scaler.transform(np.array([[y]]))[0,0]
    train_encoded.append((X, next_s, y_scaled, company_idx))

num_companies = len(enc.classes_)
print(f"✅ Encoded companies | num_companies={num_companies}")

# =====================================================
# 4️⃣ DEFINE DATASET
# =====================================================
class WindowDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        X, next_sent, y, c = self.data[idx]
        X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        next_sent = np.nan_to_num(next_sent, nan=0.0, posinf=0.0, neginf=0.0)
        y = 0.0 if not np.isfinite(y) else y

        # One-hot encode company as feature
        company_feat = np.zeros(num_companies, dtype=np.float32)
        company_feat[c] = 1.0

        return (
            torch.tensor(X, dtype=torch.float32),
            torch.tensor(next_sent, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(company_feat, dtype=torch.float32)
        )

train_loader = DataLoader(WindowDataset(train_encoded), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print(f"✅ Training samples loaded: {len(train_loader.dataset)}")

# =====================================================
# 5️⃣ DEFINE MODEL WITH DROPOUT
# =====================================================
class StockTransformer(nn.Module):
    def __init__(self, feature_dim, sentiment_dim, company_feature_dim, dropout=0.1):
        super().__init__()
        d_model = 128

        self.input_proj = nn.Linear(feature_dim + company_feature_dim, d_model)
        self.norm = nn.LayerNorm(d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.dropout = nn.Dropout(dropout)

        self.sentiment_proj = nn.Linear(sentiment_dim, 32)
        self.fc = nn.Sequential(
            nn.Linear(d_model + 32, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x, next_sent, company_feat):
        company_feat_exp = company_feat.unsqueeze(1).expand(-1, x.size(1), -1)
        x = torch.cat([x, company_feat_exp], dim=-1)

        x = self.input_proj(x)
        x = self.norm(x)

        x = self.encoder(x)
        x = self.dropout(x)

        seq_out = x[:, -1, :]
        sent_embed = self.sentiment_proj(next_sent)
        combined = torch.cat([seq_out, sent_embed], dim=-1)
        return self.fc(combined).squeeze(-1)

# =====================================================
# 6️⃣ INITIALIZE MODEL, OPTIMIZER, LOSS
# =====================================================
feature_dim = train_encoded[0][0].shape[1]
sentiment_dim = train_encoded[0][1].shape[0]

model = StockTransformer(feature_dim, sentiment_dim, num_companies, dropout=DROPOUT).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss()  # Huber loss

print(f"✅ Model initialized | feature_dim={feature_dim}, sentiment_dim={sentiment_dim}, companies={num_companies}")

# =====================================================
# 7️⃣ TRAINING LOOP
# =====================================================
for epoch in range(EPOCHS):
    model.train()
    total_loss, skipped = 0.0, 0

    for X, next_sent, y, c in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        X, next_sent, y, c = X.to(DEVICE), next_sent.to(DEVICE), y.to(DEVICE), c.to(DEVICE)
        opt.zero_grad()

        pred = model(X, next_sent, c)

        if torch.isnan(pred).any() or torch.isinf(pred).any():
            skipped += 1
            continue

        loss = loss_fn(pred, y)
        if torch.isnan(loss) or torch.isinf(loss):
            skipped += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        total_loss += loss.item()

    avg_loss = total_loss / (len(train_loader) - skipped + 1e-8)
    print(f"Epoch {epoch+1:03d} | Train Loss: {avg_loss:.6f} | Skipped: {skipped}")

# =====================================================
# 8️⃣ SAVE MODEL AND Y_SCALERS
# =====================================================
torch.save({
    "model_state": model.state_dict(),
    "companies": companies,
    "label_encoder": enc,
    "y_scalers": y_scalers  
}, MODEL_PATH)

print(f"✅ Model saved as '{MODEL_PATH}'")

🚀 Using device: cuda
✅ Loaded 'train_windows.pkl' and 'train_company_list.pkl'.
✅ Encoded companies | num_companies=8
✅ Training samples loaded: 7960
✅ Model initialized | feature_dim=12, sentiment_dim=3, companies=8


Epoch 1/500: 100%|██████████| 125/125 [00:02<00:00, 49.34it/s]


Epoch 001 | Train Loss: 0.004439 | Skipped: 0


Epoch 2/500: 100%|██████████| 125/125 [00:02<00:00, 50.92it/s]


Epoch 002 | Train Loss: 0.001651 | Skipped: 0


...

Epoch 498 | Train Loss: 0.000209 | Skipped: 0


Epoch 499/500: 100%|██████████| 125/125 [00:02<00:00, 49.94it/s]


Epoch 499 | Train Loss: 0.000206 | Skipped: 0


Epoch 500/500: 100%|██████████| 125/125 [00:02<00:00, 47.13it/s]

Epoch 500 | Train Loss: 0.000206 | Skipped: 0
✅ Model saved as 'model.pt'
